# Narrative Generator Agent – Demo (SAR & OFAC Rejected Transaction)

This notebook demonstrates the **Narrative Generator Agent** (CrewAI) for regulatory narratives:

- **SAR:** Suspicious Activity Report narrative (case, subject, alert, SuspiciousActivityInformation, transactions).
- **OFAC_REJECT:** Sanctions rejected transaction report narrative (Section 8).

The agent **routes by report type** (from input or argument), **retrieves KB guidance** (or local fallback for OFAC_REJECT), and **generates** one narrative paragraph. Output is input JSON + `{"narrative": "..."}`. **Validation** runs per report type (see Section 5 and `docs/NARRATIVE_VALIDATION.md`).

**Requirements:** Create a `.env` file in the project root with `OPENAI_API_KEY` and (for SAR) `SUPABASE_URL` and `SUPABASE_ANON_KEY`. The first setup cell loads these automatically.

## 1. Setup: Add project to path and load input example

In [1]:
from pathlib import Path
from dotenv import load_dotenv
import os

# Project root (parent of notebooks/ when running from notebooks/)
project_root = Path("..").resolve()
env_path = project_root / ".env"
load_dotenv(env_path)

# Verify keys are loaded (values are not printed for security)
has_openai = bool(os.getenv("OPENAI_API_KEY"))
has_supabase = bool(os.getenv("SUPABASE_URL")) and bool(os.getenv("SUPABASE_ANON_KEY"))
print("OPENAI_API_KEY loaded:", has_openai)
print("Supabase credentials loaded:", has_supabase)
if not has_openai:
    print("Warning: Add OPENAI_API_KEY to .env in project root to run the agent.")

OPENAI_API_KEY loaded: True
Supabase credentials loaded: True


In [ ]:
import json
import sys
from pathlib import Path

# Add project src so we can import narrative_agent
project_root = Path("..").resolve()
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
# Load the provided input example
input_path = project_root / "examples" / "input_example.json"
with open(input_path, "r") as f:
    input_example = json.load(f)
print("Case ID:", input_example["case_id"])
print("Subject:", input_example["subject"]["name"])
print("Alert red flags:", input_example["alert"]["red_flags"])
print("Transactions count:", len(input_example["transactions"]))

## 2. Few-shot examples (used by the agent)

The agent uses these examples in its prompt to learn how to translate suspicious activity data into narrative form.

In [14]:
from narrative_agent.examples import FEW_SHOT_EXAMPLES, get_few_shot_text

print(f"Number of few-shot examples: {len(FEW_SHOT_EXAMPLES)}")
for i, ex in enumerate(FEW_SHOT_EXAMPLES, 1):
    print(f"\n--- Example {i} ---")
    print("Input keys:", list(ex["input_snippet"].keys()))
    print("Narrative (first 120 chars):", ex["narrative"][:120] + "...")

Number of few-shot examples: 3

--- Example 1 ---
Input keys: ['case_id', 'amount', 'date_from', 'date_to', 'subject', 'alert', 'categories', 'activities']
Narrative (first 120 chars): 
This Suspicious Activity Report (SAR) is filed to report suspected structuring and money laundering activities involvin...

--- Example 2 ---
Input keys: ['case_id', 'amount', 'date_from', 'date_to', 'subject', 'alert', 'categories', 'activities']
Narrative (first 120 chars): 
A review of recent account activity associated with Maria Thompson identified indicators consistent with potential acco...

--- Example 3 ---
Input keys: ['case_id', 'amount', 'date_from', 'date_to', 'subject', 'alert', 'categories', 'activities']
Narrative (first 120 chars): 
Unusual international payment activity conducted by Global Imports LLC prompted further investigation into potential tr...


## 3. Reference narratives and effectiveness guidelines

The agent references these guidelines to produce narratives suitable for SAR reports.

In [15]:
from narrative_agent.narrative_reference import (
    REFERENCE_NARRATIVES,
    EFFECTIVENESS_GUIDELINES,
    get_reference_context,
)

print("Effectiveness guidelines (excerpt):")
print(EFFECTIVENESS_GUIDELINES[:500], "...")
print("\nReference narratives:")
for ref in REFERENCE_NARRATIVES:
    print(f"  - {ref['summary']}: {ref['effectiveness'][:80]}...")

Effectiveness guidelines (excerpt):

SAR narrative effectiveness guidelines (use when generating):

1. Use only information explicitly provided in the input. Do not add names, dates, amounts, or facts not present.
2. Maintain a factual and objective tone. Do not conclude that a crime occurred; describe observed activity and patterns.
3. Include specific transactional detail: dates, amounts, transaction counts, account identifiers (if provided), and total aggregates where applicable.
4. Clearly connect suspicious patterns to the un ...

Reference narratives:
  - Structured cash deposits followed by immediate international wire transfers to a single beneficiary.: Effective because it clearly documents structured deposits with specific dates a...
  - Corporate accounts exhibiting repeated structured cash deposits, high-volume wires, and foreign correspondent banking involvement.: Effective because it systematically outlines suspicious patterns including struc...
  - Grocery store owner en

## 4. Run the Narrative Generator Agent

Run the agent on the currently loaded input. The agent returns the **same JSON as input** with one new key `narrative` added.

In [5]:
# Optionally (re)load a specific input for the single run below. Default: input_example.json
input_path = project_root / "examples" / "input_example.json"
with open(input_path, "r") as f:
    input_example = json.load(f)
print("Loaded:", input_path.name, "| Case ID:", input_example["case_id"])

Loaded: input_example.json | Case ID: CASE-2024-677021


In [ ]:
from narrative_agent import generate_narrative

output = generate_narrative(input_example, verbose=True)
# Output = same as input with one new field "narrative" (all input keys preserved)
input_keys = [k for k in output.keys() if k != "narrative"]
print("\n--- Output: input + narrative (same format as input, one new field) ---")
print("Input keys preserved:", input_keys)
print("New field added: narrative")
print("\nFull output (complete JSON = input + narrative):")
print(json.dumps(output, indent=2))

In [16]:
import textwrap
from narrative_agent import generate_narrative

In [ ]:
INPUT_EXAMPLE_NAMES = [
    "input_example.json",
    # "input_example2.json",
    # "input_example3.json",
]
examples_dir = project_root / "examples"
width = 80

for filename in INPUT_EXAMPLE_NAMES:
    path = examples_dir / filename
    if not path.exists():
        print(f"Skipping {filename} (not found)")
        continue
    with open(path, "r") as f:
        data = json.load(f)
    print("=" * 60)
    print(f"Input file: {filename}")
    print(f"Case ID: {data.get('case_id', 'N/A')}")
    print("=" * 60)
    output = generate_narrative(data, verbose=False)
    # Output has same keys as input + "narrative"
    assert "narrative" in output, "Missing 'narrative' in output"
    print("Generated narrative (wrapped):")
    print("-" * 40)
    print(textwrap.fill(output["narrative"], width=width))
    print("-" * 40)
    print()

Input file: input_example.json
Case ID: CASE-2024-677021
Generated narrative (wrapped):
----------------------------------------
An investigation was initiated regarding suspicious cash deposit activity
associated with Global Trade Corp, a retail business. The suspicious conduct was
identified between March 15, 2024, and March 22, 2024, involving multiple cash
deposits structured to avoid reporting thresholds, which raised concerns about
potential money laundering. Specifically, three cash deposits were made: $9,500
on March 15, $8,000 on March 18, and another $8,000 on March 22, all originating
from the same account and followed by immediate wire transfers to different
destination accounts. The total amount involved in these transactions was
$25,500. The pattern of cash deposits and subsequent wire transfers was
inconsistent with the expected banking behavior for the subject, and the purpose
of these transactions appeared unclear, with no lawful purpose evident. The
subject declined t

## 5. Validate the narrative

Run the narrative validation checks to ensure the generated narrative passes quality criteria (structure, tone, completeness, no forbidden language). See **`docs/NARRATIVE_VALIDATION.md`** for full criteria and process.

In [17]:
from narrative_agent import validate_narrative

In [ ]:
# Validate the generated narrative against input and quality criteria
validation = validate_narrative(output["narrative"], input_example)
print("Validation passed:", validation.passed)
print()
for c in validation.checks:
    status = "PASS" if c.passed else "FAIL"
    msg = f" — {c.message}" if c.message else ""
    print(f"  [{status}] {c.name}{msg}")
if validation.failed_checks():
    print("\nFailed checks:", [c.name for c in validation.failed_checks()])
else:
    print("\nAll checks passed. Narrative is valid for SAR use.")

Validation passed: True

  [PASS] narrative_non_empty
  [PASS] no_json_in_body
  [PASS] word_count
  [PASS] no_forbidden_phrases
  [PASS] subject_mentioned — Narrative should identify or refer to the subject
  [PASS] date_mentioned — Narrative should include date or date range of activity
  [PASS] suspicious_patterns_reflected — Narrative should reflect suspicious activity types from input

All checks passed. Narrative is valid for SAR use.


In [9]:
# Full output = input + narrative (same format as input, one new field). Print so nothing is truncated:
print("Full output (input + narrative):")
print(json.dumps(output, indent=2))

Full output (input + narrative):
{
  "case_id": "CASE-2024-677021",
  "report_type": "Initial",
  "generated_at": "2025-11-18T03:42:03Z",
  "subject": {
    "subject_id": "C-94926",
    "name": "Global Trade Corp",
    "type": "Individual",
    "risk_rating": "Medium",
    "country": "US",
    "onboarding_date": "2022-05-09",
    "industry_or_occupation": "Retail",
    "beneficial_owners": [
      "Owner A (60%)",
      "Owner B (40%)"
    ],
    "prior_sars": [
      "2022-372022"
    ]
  },
  "institution": {
    "name": "Example Community Bank",
    "branch_city": "New York",
    "branch_state": "NY",
    "contact_officer": "Brian Lee",
    "contact_phone": "+1-800-XXX-XXXX",
    "primary_federal_regulator": "FDIC"
  },
  "alert": {
    "alert_id": "ALRT-78051",
    "alert_date": "2025-01-11",
    "model_name": "ML Risk Model v2",
    "trigger_reasons": [
      "Unusual cash pattern",
      "Multiple day deposits"
    ],
    "risk_score": 0.41,
    "red_flags": [
      "Structuring"

## 6. Display the generated narrative

The narrative is the mandatory section content for the SAR report. It is available as `output["narrative"]`; the rest of `output` is the original input.

In [10]:
import textwrap

narrative_text = output["narrative"]
width = 80
print("Generated SAR narrative:")
print("-" * 60)
print(textwrap.fill(narrative_text, width=width))
print("-" * 60)

Generated SAR narrative:
------------------------------------------------------------
An investigation was initiated regarding suspicious cash deposit activity
associated with Global Trade Corp, a retail business. The suspicious conduct was
identified between March 15, 2024, and March 22, 2024, involving multiple cash
deposits structured to avoid reporting thresholds, which raised concerns about
potential money laundering. Specifically, three cash deposits were made: $9,500
on March 15, $8,000 on March 18, and another $8,000 on March 22, all originating
from the same account and followed by immediate wire transfers to different
destination accounts. The total amount involved in these transactions was
$25,500. The pattern of cash deposits and subsequent wire transfers was
inconsistent with the expected banking behavior for the subject, and the purpose
of these transactions appeared unclear, with no lawful purpose evident. The
subject declined to provide explanations for the large cash d

## 7. Test with all 3 input examples

Run the agent on each of the three example JSON files: generate narrative, validate it, and display the result for each.

In [10]:
import textwrap
from narrative_agent import generate_narrative, validate_narrative

In [ ]:
INPUT_EXAMPLE_NAMES = [
    "input_example.json",
    "input_example2.json",
    "input_example3.json",
]
examples_dir = project_root / "examples"
width = 80

for filename in INPUT_EXAMPLE_NAMES:
    path = examples_dir / filename
    if not path.exists():
        print(f"Skipping {filename} (not found)")
        continue
    with open(path, "r") as f:
        data = json.load(f)
    print("=" * 60)
    print(f"Input file: {filename}")
    print(f"Case ID: {data.get('case_id', 'N/A')}")
    print("=" * 60)
    output = generate_narrative(data, verbose=False)
    assert "narrative" in output, "Missing 'narrative' in output"
    validation = validate_narrative(output["narrative"], data)
    print("Generated narrative (wrapped):")
    print("-" * 40)
    print(textwrap.fill(output["narrative"], width=width))
    print("-" * 40)
    print(
        "Validation:",
        "PASSED" if validation.passed else "FAILED",
        (
            list(c.name for c in validation.failed_checks())
            if not validation.passed
            else ""
        ),
    )
    print()

Input file: input_example.json
Case ID: CASE-2024-677021
Generated narrative (wrapped):
----------------------------------------
An investigation was initiated regarding suspicious cash deposit activity
associated with Global Trade Corp, a retail business. The suspicious activity
was identified between March 15, 2024, and March 22, 2024, involving multiple
cash deposits that appeared to be structured to evade reporting requirements.
Specifically, three cash deposits were made: the first on March 15, 2024, for
$9,500, followed by another on March 18, 2024, for $8,000, and a final deposit
on March 22, 2024, also for $8,000. Each of these transactions was conducted at
a location in Miami, FL, and was followed by immediate wire transfers to
different accounts, totaling $25,500. The transaction patterns raised concerns
due to their rapid movement of funds and the use of multiple accounts, which are
indicative of potential money laundering activity. Notably, the subject declined
to provide a

In [12]:
output

{'case_id': 'CASE-2025-425659',
 'report_type': 'Initial',
 'generated_at': '2025-11-18T03:41:19Z',
 'subject': {'subject_id': 'C-72530',
  'name': 'Global Trade Corp',
  'type': 'Business',
  'risk_rating': 'Low',
  'country': 'US',
  'onboarding_date': '2017-10-25',
  'industry_or_occupation': 'Import/Export',
  'beneficial_owners': ['Owner A (60%)', 'Owner B (40%)'],
  'prior_sars': ['2022-636221']},
 'institution': {'name': 'Example Community Bank',
  'branch_city': 'Charlotte',
  'branch_state': 'NY',
  'contact_officer': 'Brian Lee',
  'contact_phone': '+1-800-XXX-XXXX',
  'primary_federal_regulator': 'NCUA'},
 'alert': {'alert_id': 'ALRT-68434',
  'alert_date': '2024-04-13',
  'model_name': 'ML Risk Model v2',
  'trigger_reasons': ['Unusual cash pattern', 'Multiple day deposits'],
  'risk_score': 0.7,
  'red_flags': ['Structuring',
   'Multiple transactions below CTR threshold',
   'Suspicious structuring',
   'Transaction(s) out of pattern',
   'No apparent business purpose']},

## 8. OFAC Sanctions Rejected Transaction Report

The same agent supports **OFAC_REJECT**: narratives for sanctions rejected transaction reports. The flow is the same: **detect report type from input** → **retrieve KB guidance** (or local fallback for OFAC_REJECT) → **generate narrative** → **validate**.

- **Required input keys:** `case_id`, `transaction`, `case_facts` (and optionally `report_type_code: "OFAC_REJECT"`).
- The narrative should summarize the attempted transaction, explain what triggered sanctions review, describe the sanctions nexus, state why the transaction was rejected, mention documents reviewed, and clearly state the transaction was **not processed** (rejected), without confusing rejection with blocking.

In [2]:
import json
import sys
from pathlib import Path

# Add project src so we can import narrative_agent
project_root = Path("..").resolve()
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
# Load the provided input example
input_path = project_root / "examples" / "input_example.json"
with open(input_path, "r") as f:
    input_example = json.load(f)
print("Case ID:", input_example["case_id"])
print("Subject:", input_example["subject"]["name"])
print("Alert red flags:", input_example["alert"]["red_flags"])
print("Transactions count:", len(input_example["transactions"]))

Case ID: CASE-2024-677021
Subject: Global Trade Corp
Alert red flags: ['Structuring']
Transactions count: 3


In [3]:
import textwrap
from narrative_agent import generate_narrative, validate_narrative
import json
import sys
from pathlib import Path

In [4]:
# Load OFAC rejected transaction example (report_type_code in input routes to OFAC flow)
ofac_path = project_root / "examples" / "ofac_reject_example.json"
with open(ofac_path, "r") as f:
    ofac_input = json.load(f)

print(
    "Case ID:",
    ofac_input["case_id"],
    "| Report type:",
    ofac_input.get("report_type_code", "N/A"),
)
print(
    "Transaction:",
    ofac_input["transaction"].get("transaction_type"),
    ofac_input["transaction"].get("amount_rejected"),
    ofac_input["transaction"].get("currency"),
)
print()

# Generate narrative (same API; routing by report_type_code from input)
ofac_output = generate_narrative(ofac_input, verbose=False)
assert "narrative" in ofac_output

# Validate OFAC narrative
ofac_validation = validate_narrative(
    ofac_output["narrative"], ofac_input, report_type_code="OFAC_REJECT"
)
print("Validation passed:", ofac_validation.passed)
for c in ofac_validation.checks:
    print("  [PASS]" if c.passed else "  [FAIL]", c.name, c.message or "")

print("\n--- Generated OFAC narrative (wrapped) ---")
print(textwrap.fill(ofac_output["narrative"], width=80))

Case ID: OFAC-SYN-001 | Report type: OFAC_REJECT
Transaction: Outbound wire transfer 18450.00 USD

Validation passed: True
  [PASS] narrative_non_empty 
  [PASS] no_json_in_body 
  [PASS] word_count 
  [PASS] no_forbidden_phrases 
  [PASS] rejection_not_blocking 
  [PASS] rejection_stated Narrative should clearly state the transaction was rejected / not processed
  [PASS] sanctions_nexus_mentioned Narrative should describe the sanctions nexus
  [PASS] documents_reviewed_mentioned Narrative should mention documents or materials reviewed

--- Generated OFAC narrative (wrapped) ---
On March 6, 2026, the institution identified an attempted outbound wire transfer
in the amount of USD 18,450 involving ABC Import Services as originator and
Tehran Industrial Supply Co. as beneficiary, with the payment reference
indicating settlement for industrial components. The transaction generated a
sanctions alert during automated screening due to the beneficiary's location in
Iran, a comprehensively sanc

In [5]:
ofac_output

{'case_id': 'OFAC-SYN-001',
 'report_type_code': 'OFAC_REJECT',
 'scenario_type': 'sanctioned_jurisdiction',
 'sanctions_program': 'Iran — ITSR',
 'institution': {'name': 'First National Bank',
  'type': 'Bank',
  'address': '100 Commerce Street',
  'city': 'Houston',
  'state': 'TX',
  'postal_code': '77002',
  'country': 'United States',
  'contact_person': 'Sandra Reyes',
  'telephone': '713-555-0142',
  'email': 'sreyes@fnbhouston.com',
  'fax': '713-555-0199'},
 'transaction': {'amount_rejected': '18450.00',
  'currency': 'USD',
  'transaction_type': 'Outbound wire transfer',
  'date_of_transaction': '2026-03-06',
  'date_of_rejection': '2026-03-06',
  'program_or_reason': 'Iran — ITSR (Iran Transactions and Sanctions Regulations), 31 C.F.R. Part 560',
  'originator_name_address': 'ABC Import Services, 4210 Industrial Blvd, Houston, TX 77001, United States',
  'originating_fi': 'First National Bank, 100 Commerce Street, Houston, TX 77002, United States',
  'intermediary_fi': 'Citi